# 03 · Model Training
Train four baseline algorithms on the same stratified split. Preprocessing is inside each sklearn Pipeline so it learns only from training data. Class weighting addresses imbalance where supported; KNN has no class_weight parameter.


In [ ]:
%pip -q install pandas numpy scikit-learn joblib
from pathlib import Path
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
try:
 from google.colab import drive,files
 drive.mount('/content/drive')
except ImportError: files=None
ROOT=Path('/content/drive/MyDrive/insurance_fraud_ml'); (ROOT/'data').mkdir(parents=True,exist_ok=True); (ROOT/'models').mkdir(exist_ok=True); (ROOT/'results').mkdir(exist_ok=True)
paths=[p for p in (ROOT/'data').glob('*.csv') if p.name not in ['X_train.csv','X_test.csv','y_train.csv','y_test.csv']]
if paths: csv=paths[0]
elif files:
 print('Upload original CSV'); up=files.upload()
 if not up: raise FileNotFoundError('Upload CSV')
 csv=Path(next(iter(up)))
else: csv=Path(input('CSV path: '))
df=pd.read_csv(csv); t='FraudFound_P'; df[t]=pd.to_numeric(df[t],errors='coerce'); df=df.dropna(subset=[t]); df[t]=df[t].astype(int)
X=df.drop(columns=[t]); y=df[t]; ids=[c for c in X if any(k in c.lower() for k in ['policy_number','claim_id','record_id','customer_id'])]; X=X.drop(columns=ids)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y); nums=Xtr.select_dtypes(include='number').columns.tolist(); cats=Xtr.select_dtypes(exclude='number').columns.tolist()
def prep(scale=False):
 n=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True))]+([('scale',StandardScaler())] if scale else []))
 c=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore',min_frequency=2))])
 return ColumnTransformer([('num',n,nums),('cat',c,cats)])
models={'Logistic Regression':Pipeline([('prep',prep(True)),('model',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42))]),'Decision Tree':Pipeline([('prep',prep()),('model',DecisionTreeClassifier(class_weight='balanced',min_samples_leaf=5,random_state=42))]),'Random Forest':Pipeline([('prep',prep()),('model',RandomForestClassifier(n_estimators=300,class_weight='balanced_subsample',min_samples_leaf=2,n_jobs=-1,random_state=42))]),'K-Nearest Neighbors':Pipeline([('prep',prep(True)),('model',KNeighborsClassifier(n_neighbors=5,weights='distance',n_jobs=-1))]}
rows=[]
for name,m in models.items():
 m.fit(Xtr,ytr); p=m.predict(Xte); rows.append({'model':name,'train_accuracy':m.score(Xtr,ytr),'test_accuracy':accuracy_score(yte,p),'test_precision':precision_score(yte,p,zero_division=0),'test_recall':recall_score(yte,p,zero_division=0),'test_f1':f1_score(yte,p,zero_division=0)}); print(name,'training complete')
comparison=pd.DataFrame(rows); display(comparison); joblib.dump({'models':models,'X_test':Xte,'y_test':yte},ROOT/'models/trained_models.joblib'); comparison.to_csv(ROOT/'results/training_comparison.csv',index=False)


Logistic Regression estimates class probabilities from weighted features. A Decision Tree learns rule-like partitions. Random Forest averages many randomized trees. KNN predicts from nearby samples and is sensitive to feature scale. Training accuracy is included for comparison, but held-out metrics are more informative. These are untuned baselines; test data must not be used for tuning.